In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf
from google.colab import files
import os
import base64
from io import BytesIO
from datetime import datetime
from IPython.display import HTML  # Added for displaying HTML in Colab

# Lista de archivos esperados
files_expected = [
    'APPLE 1 MINUTO 29-30.xlsx',
    'APPLE 5 MINUTOS 29-30.xlsx',
    'CHEVRON 1 MINUTO 29-30.xlsx',
    'CHEVRON 5 MINUTOS 29-30.xlsx',
    'JPMORGAN 1 MINUTO 29-30.xlsx',
    'JPMORGAN 5 MINUTOS 29-30.xlsx'
]

# Verificar y cargar archivos
uploaded = {}
for file_name in files_expected:
    if not os.path.exists(file_name):
        print(f"Subiendo {file_name}...")
        uploaded.update(files.upload())
    else:
        uploaded[file_name] = file_name

# Función para seleccionar la primera columna numérica válida
def select_numeric_column(df):
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) and not pd.api.types.is_datetime64_any_dtype(df[col]):
            return col
    raise ValueError("No se encontró una columna numérica válida")

# Función para guardar gráfica como base64
def save_plot_base64():
    buffer = BytesIO()
    plt.savefig(buffer, format='png', bbox_inches='tight')
    buffer.seek(0)
    img_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
    plt.close()
    return img_str

# Inicializar contenido HTML con llaves escapadas
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Análisis de Series de Tiempo</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        h1, h2, h3 {{ color: #333; }}
        img {{ max-width: 100%; height: auto; }}
        table {{ border-collapse: collapse; width: 100%; margin-bottom: 20px; }}
        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        th {{ background-color: #f2f2f2; }}
    </style>
</head>
<body>
    <h1>Análisis de Series de Tiempo</h1>
    <p>Fecha de generación: {}</p>
""".format(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Procesar cada serie de tiempo
data_dict = {}
for file_name in uploaded.keys():
    df = pd.read_excel(file_name)
    series_name = file_name.replace('.xlsx', '')

    # Seleccionar columna numérica
    num_col = select_numeric_column(df)
    series = df[num_col].dropna()
    data_dict[series_name] = series

    html_content += f"<h2>Análisis de {series_name}</h2>"

    # Gráfico de la serie original
    plt.figure(figsize=(10, 4))
    plt.plot(series, label='Serie Original')
    plt.title(f'Serie de Tiempo: {series_name}')
    plt.xlabel('Índice')
    plt.ylabel(num_col)
    plt.legend()
    plt.grid(True)
    html_content += f'<h3>Serie Original</h3><img src="data:image/png;base64,{save_plot_base64()}">'

    # Prueba ADF en serie original
    adf_result = adfuller(series)
    html_content += f"""
    <h3>Prueba ADF (Serie Original)</h3>
    <table>
        <tr><th>Estadístico</th><td>{adf_result[0]:.4f}</td></tr>
        <tr><th>P-valor</th><td>{adf_result[1]:.4f}</td></tr>
        <tr><th>Valores Críticos</th><td>{', '.join([f'{k}: {v:.4f}' for k, v in adf_result[4].items()])}</td></tr>
    </table>
    """

    # Prueba ADF en serie diferenciada
    diff_series = series.diff().dropna()
    adf_diff_result = adfuller(diff_series)
    html_content += f"""
    <h3>Prueba ADF (Serie Diferenciada)</h3>
    <table>
        <tr><th>Estadístico</th><td>{adf_diff_result[0]:.4f}</td></tr>
        <tr><th>P-valor</th><td>{adf_diff_result[1]:.4f}</td></tr>
        <tr><th>Valores Críticos</th><td>{', '.join([f'{k}: {v:.4f}' for k, v in adf_diff_result[4].items()])}</td></tr>
    </table>
    """

    # Modelo ARIMA(1,0,1)
    try:
        model = ARIMA(series, order=(1,0,1))
        model_fit = model.fit()
        residuals = model_fit.resid

        # Gráfico de residuales
        plt.figure(figsize=(10, 4))
        plt.plot(residuals, label='Residuales')
        plt.title(f'Residuales ARIMA(1,0,1): {series_name}')
        plt.xlabel('Índice')
        plt.ylabel('Residuales')
        plt.legend()
        plt.grid(True)
        html_content += f'<h3>Residuales ARIMA(1,0,1)</h3><img src="data:image/png;base64,{save_plot_base64()}">'

        # Gráfico ACF de residuales
        plt.figure(figsize=(10, 4))
        plot_acf(residuals, lags=20, ax=plt.gca())
        plt.title(f'ACF de Residuales: {series_name}')
        html_content += f'<h3>ACF de Residuales</h3><img src="data:image/png;base64,{save_plot_base64()}">'
    except Exception as e:
        html_content += f"<h3>Error en ARIMA para {series_name}</h3><p>{str(e)}</p>"

# Pruebas de cointegración
html_content += "<h2>Pruebas de Cointegración</h2>"
pairs = [
    ('APPLE 1 MINUTO 29-30', 'CHEVRON 1 MINUTO 29-30'),
    ('APPLE 1 MINUTO 29-30', 'JPMORGAN 1 MINUTO 29-30'),
    ('CHEVRON 1 MINUTO 29-30', 'JPMORGAN 1 MINUTO 29-30'),
    ('APPLE 5 MINUTOS 29-30', 'CHEVRON 5 MINUTOS 29-30'),
    ('APPLE 5 MINUTOS 29-30', 'JPMORGAN 5 MINUTOS 29-30'),
    ('CHEVRON 5 MINUTOS 29-30', 'JPMORGAN 5 MINUTOS 29-30')
]

for pair in pairs:
    try:
        series1, series2 = data_dict[pair[0]], data_dict[pair[1]]
        # Asegurar misma longitud
        min_len = min(len(series1), len(series2))
        series1, series2 = series1[:min_len], series2[:min_len]

        score, p_value, _ = coint(series1, series2)
        html_content += f"""
        <h3>Cointegración: {pair[0]} vs {pair[1]}</h3>
        <table>
            <tr><th>Estadístico</th><td>{score:.4f}</td></tr>
            <tr><th>P-valor</th><td>{p_value:.4f}</td></tr>
        </table>
        """
    except Exception as e:
        html_content += f"<h3>Error en Cointegración {pair[0]} vs {pair[1]}</h3><p>{str(e)}</p>"

# Finalizar HTML
html_content += "</body></html>"

# Mostrar HTML en Colab
display(HTML(html_content))

# Guardar archivo HTML
with open('time_series_analysis.html', 'w') as f:
    f.write(html_content)

# Descargar archivo HTML
files.download('time_series_analysis.html')